In [1]:
%pip install -U selenium

Note: you may need to restart the kernel to use updated packages.


In [4]:
%run everland_reviews_latest

Exception: File `'everland_reviews_latest'` not found.

In [8]:
# -*- coding: utf-8 -*-
"""에버랜드 방문자 리뷰 — 최신순, 방문일 2024-01-01 ~ 실행일(한국 시간).

처음 한 번(주피터 셀): %pip install -U selenium
다음 셀: %run everland_reviews_latest.py
터미널: python everland_reviews_latest.py
날짜 고정: python everland_reviews_latest.py --end 2026-10-22

결과는 이 파일 옆 everland_output 폴더에 저장합니다.
* everland_reviews.csv: 요청한 9개 열, 방문일 내림차순, UTF-8 BOM
* checkpoint.sqlite3: 매 묶음 확정 저장. 재실행해도 중복을 줄여 이어 모음
* review_status.json / collection.log: 실제 저장 수, 종료/오류 상태
* date_unresolved.csv: 연도를 확인할 수 없는 리뷰(연도 추측하지 않음)

빈 본문/평점/선택항목도 그대로 빈칸입니다. 아이디는 공개 닉네임,
결제유형은 화면의 인증 수단이며 실제 결제 카드/현금 종류가 아닙니다.
재방문은 화면의 'N번째 방문' 원문입니다. 감성분석으로 항목을 채우지 않습니다.

최신순 화면이 방문일로 정렬된다는 보장은 없으므로 기본은 두 목록을
끝까지 탐색하고, CSV에는 대상 방문일만 저장합니다. 오래된 방문일 하나를
만나도 종료하지 않습니다. --boundary-stop은 연속 3묶음이 기간보다 오래된
경우 조기 종료하는 선택 기능이며, 전체 수집 여부는 확인되지 않습니다.

일시적인 브라우저 오류 때 60초/120초/240초 후 최대 3번 재시작합니다.
다음 리뷰 묶음은 기본 10초 간격으로 요청합니다. 이는 네이버가 허용한 속도가
아니며, 제한 방지를 보장하지 않습니다. 화면 재탐색은 최신부터 시작하지만 저장한
행은 유지합니다. 접속 차단/인증 화면은 우회하지 않고 현재 결과를 저장합니다.
CSV를 엑셀로 열어 잠그면 *_backup.csv로 대신 저장합니다.
Windows 실행 중에는 일시적으로 자동 절전을 억제하고 종료 때 해제합니다.
컴퓨터 종료/뚜껑 닫기/인터넷 단절/저장장치 고장까지 방지할 수는 없습니다.

현재 화면 선택자 확인일: 2026-10-22. 네이버 화면 개편 시 수정 필요.
모든 기간에 대한 실수집 완료 검증은 하지 않았습니다.
"""

import argparse
import csv
import ctypes
import hashlib
import json
import logging
import os
import re
import sqlite3
import sys
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

KST = timezone(timedelta(hours=9))
URL = "https://map.naver.com/p/entry/place/11574457?placePath=%2Freview"
COLUMNS = ["아이디", "날짜", "게시글", "평점", "결제유형", "방문유형", "동행자", "재방문", "키워드 리뷰"]
BASE = Path.cwd()
PURPOSES = {"데이트", "가족모임", "나들이", "친목", "여행", "일상", "기념일", "회식", "비즈니스", "친목모임", "단체모임"}
COMPANIONS = {"혼자", "아이", "친구", "부모님", "연인・배우자", "연인·배우자", "연인", "배우자", "친척・형제자매", "친척·형제자매", "동료", "지인", "반려동물"}


def now():
    return datetime.now(KST).isoformat(timespec="seconds")


def visit_date(raw):
    """전체 연도만 사용. 10.21 같은 표기에서 연도를 추정하지 않습니다."""
    for pattern in (r"(?<!\d)(20\d{2})\s*년\s*(\d{1,2})\s*월\s*(\d{1,2})\s*일",
                    r"(?<!\d)(20\d{2})[./-](\d{1,2})[./-](\d{1,2})(?!\d)"):
        m = re.search(pattern, raw or "")
        if m:
            try:
                return date(*map(int, m.groups())).isoformat()
            except ValueError:
                return ""
    return ""


def parse(raw):
    purposes, companions, unknown = [], [], []
    for item in raw.get("visit", []):
        item = item.strip()
        if not item or item.startswith(("대기 시간", "예약 후", "예약 없이")):
            continue
        tokens = [x.strip() for x in item.split(",") if x.strip()]
        if tokens and all(x in PURPOSES for x in tokens):
            purposes.extend(tokens)
        elif tokens and all(x in COMPANIONS for x in tokens):
            companions.extend(tokens)
        else:
            unknown.append(item)
    rating = ""
    m = re.search(r"(?:별점\s*)?([0-5](?:\.\d+)?)\s*(?:점)?$", raw.get("rating", "").strip())
    if m and 0 <= float(m.group(1)) <= 5:
        rating = m.group(1)
    repeat = re.search(r"\d+\s*번째\s*방문", raw.get("meta", ""))
    row = dict(zip(COLUMNS, [
        raw.get("nickname", "").strip(), visit_date(raw.get("date", "")),
        raw.get("content", "").strip(), rating, raw.get("verification", "").strip(),
        " | ".join(dict.fromkeys(purposes)), " | ".join(dict.fromkeys(companions)),
        repeat.group(0) if repeat else "", " | ".join(dict.fromkeys(raw.get("keywords", [])))
    ]))
    return row, unknown


def atomic_text(path, text):
    temp = path.with_name(path.name + ".tmp")
    with temp.open("w", encoding="utf-8") as f:
        f.write(text)
        f.flush()
        os.fsync(f.fileno())
    os.replace(temp, path)


def write_csv(path, rows, columns=COLUMNS):
    """기존 CSV를 먼저 지우지 않음. 새 파일을 모두 쓴 다음 교체."""
    temp = path.with_name(path.name + ".tmp")
    with temp.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
        f.flush()
        os.fsync(f.fileno())
    try:
        os.replace(temp, path)
        return path
    except PermissionError:
        backup = path.with_name(path.stem + "_backup.csv")
        try:
            os.replace(temp, backup)
        except PermissionError:
            backup = path.with_name(path.stem + "_backup_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f") + ".csv")
            os.replace(temp, backup)
        logging.warning("CSV가 열려 있어 대체 파일에 저장: %s", backup)
        return backup


class Store:
    def __init__(self, folder, start, end):
        self.folder, self.start, self.end = Path(folder), start, end
        self.folder.mkdir(parents=True, exist_ok=True)
        self.db = sqlite3.connect(self.folder / "checkpoint.sqlite3")
        self.db.execute("PRAGMA synchronous=FULL")
        self.db.execute("CREATE TABLE IF NOT EXISTS reviews (uid TEXT PRIMARY KEY, visit TEXT, row_json TEXT, raw_json TEXT, unknown TEXT)")
        self.state = {"시작": now(), "시작일": start, "종료일": end, "상태": "실행 준비",
                      "목록": {}, "전체수집보장": False,
                      "안내": "공개 화면에서 확인한 리뷰만 수집. 비공개/삭제/미제공 리뷰는 확인 불가. 동일 프로필·날짜·본문·선택정보의 동일 행은 중복으로 처리."}

    def add(self, records):
        with self.db:
            for raw in records:
                row, unknown = parse(raw)
                # 내용/방문정보까지 포함. 닉네임만으로 중복 판정하지 않습니다.
                identity = [raw.get("profile", ""), row, raw.get("date", "")]
                uid = hashlib.sha256(json.dumps(identity, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
                self.db.execute("INSERT OR REPLACE INTO reviews VALUES(?,?,?,?,?)", (
                    uid, row["날짜"], json.dumps(row, ensure_ascii=False),
                    json.dumps(raw, ensure_ascii=False), json.dumps(unknown, ensure_ascii=False)))

    def save(self, state=None):
        if state:
            self.state["상태"] = state
        rows = [json.loads(x[0]) for x in self.db.execute(
            "SELECT row_json FROM reviews WHERE visit >= ? AND visit <= ? ORDER BY visit DESC, rowid ASC",
            (self.start, self.end))]
        actual = write_csv(self.folder / "everland_reviews.csv", rows)
        unresolved = []
        for row_json, raw_json in self.db.execute("SELECT row_json, raw_json FROM reviews WHERE visit = ''"):
            row, raw = json.loads(row_json), json.loads(raw_json)
            row["날짜원문"] = raw.get("date", "")
            unresolved.append(row)
        write_csv(self.folder / "date_unresolved.csv", unresolved, COLUMNS + ["날짜원문"])
        self.state.update({"마지막저장": now(), "CSV파일": str(actual.resolve()), "저장건수": len(rows),
                           "날짜확인필요": len(unresolved),
                           "선택항목확인필요": self.db.execute("SELECT count(*) FROM reviews WHERE unknown != '[]'").fetchone()[0],
                           "가장최근방문일": rows[0]["날짜"] if rows else None,
                           "가장오래된방문일": rows[-1]["날짜"] if rows else None})
        atomic_text(self.folder / "review_status.json", json.dumps(self.state, ensure_ascii=False, indent=2))
        return len(rows)


# 2026-10-22 실제 공개 리뷰 DOM에서 확인한 선택자.
SECTION_JS = """
const kind = arguments[0];
const section = kind === '본문'
 ? document.querySelector('#_review_list')?.closest('.place_section')
 : Array.from(document.querySelectorAll('.place_section')).find(x =>
     x.querySelector('h2')?.textContent.includes('키워드·별점 리뷰'));
"""
SNAPSHOT_JS = SECTION_JS + """
if (!section) return {missing:true, count:0, records:[], more:false};
const all = Array.from(section.querySelectorAll('li.place_apply_pui'));
function txt(el) { return (el?.textContent || '').trim(); }
function contentText(el) {
  if (!el) return '';
  function walk(n) {
    if (n.nodeType === 3) return n.nodeValue;
    if (n.nodeName === 'BR') return '\\n';
    return Array.from(n.childNodes).map(walk).join('');
  }
  return walk(el).trim();
}
return {missing:false, count:all.length,
  more:!!section.querySelector('[data-nlog-area="plc_rrv.more"], [data-nlog-area="plc_rrv.starmore"]'),
  records:all.slice(arguments[1]).map(card => {
    const meta = card.querySelector('.pui__QKE5Pr');
    const dateNode = meta?.querySelector('time');
    const parts = Array.from(meta?.children || []);
    const verify = parts.find(n => n.textContent.includes('인증 수단'));
    return {
      nickname:txt(card.querySelector('.pui__NMi-Dp')),
      profile:card.querySelector('[data-pui-click-code="profile"]')?.getAttribute('href') || '',
      date:txt(dateNode?.parentElement), meta:txt(meta),
      verification:txt(verify).replace('인증 수단','').trim(),
      rating:txt(card.querySelector('.pui__6abRMf')),
      content:contentText(card.querySelector('[data-pui-click-code="rvshowmore"], [data-pui-click-code="rvshowless"]')),
      visit:Array.from(card.querySelectorAll('[data-pui-click-code="visitkeywords"] > span')).map(txt),
      keywords:Array.from(card.querySelectorAll('li.pui__jhpEyP')).map(txt)
    };
  })};
"""


class AccessBlocked(RuntimeError):
    pass


class RetryableReviewError(RuntimeError):
    """리뷰 응답 지연처럼 저장 후 재시도할 수 있는 오류."""
    pass


def restriction_message(text):
    """오류 안내문에서만 검사. IP라는 단어 자체는 제한 증거가 아닙니다."""
    text = re.sub(r"\s+", " ", text or "").strip()
    patterns = (
        r"자동\s*입력\s*방지|비정상적인\s*(?:접근|요청|트래픽)|접근이?\s*제한",
        r"자동화된\s*요청|로봇이\s*아닙니다|Verify you are human|Access Denied",
        r"요청이?\s*차단|Too Many Requests|이용이\s*(?:일시적으로\s*)?제한",
        r"(?:IP|아이피).{0,100}(?:차단|제한|비정상|너무\s*많은)",
        r"(?:차단|제한|비정상|너무\s*많은).{0,100}(?:IP|아이피)",
    )
    return any(re.search(pattern, text, re.I) for pattern in patterns)


def check_access(driver):
    # 리뷰 본문이 아닌 페이지/경고 영역만 확인해 이용자 글의 오탐을 줄입니다.
    text = driver.execute_script("""
    const cards = document.querySelector('li.place_apply_pui');
    const alert = Array.from(document.querySelectorAll('[role="alert"], [role="dialog"], h1, h2, h3, p'))
      .filter(x=>!x.closest('li.place_apply_pui') && !x.querySelector('li.place_apply_pui'))
      .map(x=>x.innerText).join(' ');
    return cards ? alert : document.body.innerText;
    """)
    if restriction_message(text):
        raise AccessBlocked("접근 제한/인증 화면 감지: 우회하지 않고 현재 결과를 저장합니다.")


def wait_for(driver, predicate, seconds):
    """대기 중에도 제한 안내를 확인해 일반 로딩 오류로 재시작하지 않음."""
    from selenium.webdriver.support.ui import WebDriverWait
    def checked(current):
        check_access(current)
        return predicate(current)
    return WebDriverWait(driver, seconds, poll_frequency=1).until(checked)


def click(driver, element):
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", element)
    element.click()


def open_page(driver):
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    driver.get(URL)
    wait = WebDriverWait(driver, 45)
    wait_for(driver, EC.frame_to_be_available_and_switch_to_it((By.ID, "entryIframe")), 45)
    check_access(driver)
    recent = wait_for(driver, EC.element_to_be_clickable((By.XPATH, "//a[normalize-space(.)='최신순']")), 45)
    click(driver, recent)
    wait_for(driver, lambda d: "reviewSort=recent" in d.execute_script("return window.location.search"), 45)
    wait_for(driver, lambda d: d.find_elements(By.CSS_SELECTOR, "#_review_list li.place_apply_pui"), 45)
    logging.info("최신순 정렬 확인. 방문일 범위에 맞는 리뷰를 저장합니다.")


def collect_section(driver, kind, store, args):
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.common.exceptions import TimeoutException
    if kind == "키워드·별점":
        toggles = driver.find_elements(By.CSS_SELECTOR, '[data-nlog-area="plc_rrv.star"]')
        if not toggles:
            raise RuntimeError("키워드·별점 리뷰 섹션을 확인하지 못했습니다. 완료로 처리하지 않습니다.")
        if toggles[0].get_attribute("aria-expanded") != "true":
            click(driver, toggles[0])
        wait_for(driver, lambda d: d.execute_script(SNAPSHOT_JS, kind, 0)["count"] > 0, 45)
    offset, batch, old_streak = 0, 0, 0
    prev_date, monotonic = None, True
    while True:
        check_access(driver)
        snap = driver.execute_script(SNAPSHOT_JS, kind, offset)
        if snap["missing"]:
            raise RuntimeError(kind + " 리뷰 목록을 찾지 못했습니다.")
        if snap["count"] < offset:
            raise RetryableReviewError("목록 길이가 줄었습니다. 재탐색이 필요합니다.")
        records = snap["records"]
        # 키워드는 목록에서 접힐 수 있으므로 새 카드의 펼치기만 실행합니다.
        section = driver.execute_script(SECTION_JS + "return section;", kind)
        cards = section.find_elements(By.CSS_SELECTOR, 'li.place_apply_pui')[offset:]
        for card in cards:
            for selector in ('[data-pui-click-code="keywordmore"]',):
                expanders = card.find_elements(By.CSS_SELECTOR, selector)
                if expanders and expanders[0].is_displayed():
                    click(driver, expanders[0])
                    time.sleep(1)
        if cards:
            records = driver.execute_script(SNAPSHOT_JS, kind, offset)["records"]
        if records:
            store.add(records)  # CSV보다 먼저 디스크에 커밋합니다.
            dates = [visit_date(r.get("date", "")) for r in records]
            for d in dates:
                if not d:
                    monotonic = False
                else:
                    if prev_date and d > prev_date:
                        monotonic = False
                    prev_date = d
            old_streak = old_streak + 1 if dates and all(d and d < args.start for d in dates) else 0
            offset, batch = snap["count"], batch + 1
            store.state["목록"][kind] = {"상태": "수집 중", "확인카드": offset, "마지막방문일": dates[-1]}
            saved = store.save("수집 중")
            logging.info("%s | 확인 %s개 | 기간 내 CSV %s개 저장 | 현재 방문일 %s", kind, offset, saved, dates[-1] or "확인 불가")
            if args.boundary_stop and monotonic and old_streak >= 3:
                store.state["목록"][kind]["상태"] = "날짜 경계로 조기 종료(누락 여부 미확인)"
                store.save()
                return
        more = driver.execute_script(SECTION_JS + "return section?.querySelector('[data-nlog-area=\"plc_rrv.more\"], [data-nlog-area=\"plc_rrv.starmore\"]');", kind)
        if more is None:
            # 로딩 중 버튼 소실을 끝으로 오인하지 않도록 두 번 확인합니다.
            time.sleep(3)
            check_access(driver)
            after = driver.execute_script(SNAPSHOT_JS, kind, offset)
            if after["count"] > offset or after["more"]:
                continue
            if after["missing"] or after["count"] == 0:
                raise RuntimeError("목록이 비어 전체 확인 여부를 판단할 수 없습니다.")
            store.state["목록"][kind]["상태"] = "화면 목록 끝까지 확인"
            store.save()
            return
        time.sleep(max(1.5, args.delay))
        before = offset
        click(driver, more)
        try:
            wait_for(driver, lambda d: d.execute_script(SNAPSHOT_JS, kind, before)["count"] > before, 45)
        except TimeoutException:
            check_access(driver)
            # 늦은 응답을 한 번 더 기다립니다. 같은 더보기를 연타하지 않습니다.
            try:
                wait_for(driver, lambda d: d.execute_script(SNAPSHOT_JS, kind, before)["count"] > before, 30)
            except TimeoutException:
                final_snap = driver.execute_script(SNAPSHOT_JS, kind, before)
                if not final_snap["more"] and final_snap["count"] >= before:
                    store.state["목록"][kind]["상태"] = "화면 목록 끝까지 확인"
                    store.save()
                    return
                raise RetryableReviewError("더보기 뒤 새 리뷰가 75초 동안 추가되지 않았습니다.")


def prevent_sleep(enable):
    if sys.platform == "win32":
        try:
            result = ctypes.windll.kernel32.SetThreadExecutionState(0x80000001 if enable else 0x80000000)
            if enable and not result:
                logging.warning("절전 억제 실패: Windows 전원 설정에서 절전을 직접 꺼 주세요.")
        except Exception:
            logging.warning("절전 억제를 적용하지 못했습니다.")


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--start", default="2024-01-01")
    parser.add_argument("--end", default=datetime.now(KST).date().isoformat())
    parser.add_argument("--output", type=Path, default=BASE / "everland_output")
    parser.add_argument("--delay", type=float, default=10.0)
    parser.add_argument("--restarts", type=int, default=3)
    parser.add_argument("--boundary-stop", action="store_true", help="기간 이전 3묶음으로 조기 종료. 누락 가능성이 있어 기본 비활성화.")
    parser.add_argument("--export-only", action="store_true", help="브라우저 없이 체크포인트에서 CSV 복원")
    args = parser.parse_args([])
    if date.fromisoformat(args.start) > date.fromisoformat(args.end):
        parser.error("시작일은 종료일보다 늦을 수 없습니다.")
    args.output.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",
                        handlers=[logging.StreamHandler(), logging.FileHandler(args.output / "collection.log", encoding="utf-8")], force=True)
    store, driver, done = Store(args.output, args.start, args.end), None, set()
    state = "시작 전"
    prevent_sleep(True)
    try:
        count = store.save("기존 수집분 복원")
        logging.info("저장 폴더: %s | 기존 기간 내 리뷰: %d", args.output.resolve(), count)
        if args.export_only:
            state = "체크포인트에서 CSV 복원 완료(수집 실행 아님)"
            return
        from selenium import webdriver
        from selenium.webdriver.chrome.options import Options
        from selenium.common.exceptions import (
            TimeoutException, StaleElementReferenceException,
            ElementClickInterceptedException, InvalidSessionIdException,
            NoSuchWindowException, WebDriverException,
        )
        transient_errors = (RetryableReviewError, TimeoutException, StaleElementReferenceException,
                            ElementClickInterceptedException, InvalidSessionIdException, NoSuchWindowException)
        for attempt in range(max(0, args.restarts) + 1):
            try:
                options = Options()
                options.add_argument("--lang=ko-KR")
                options.add_argument("--window-size=1440,1000")
                driver = webdriver.Chrome(options=options)
                driver.set_page_load_timeout(60)
                driver.set_script_timeout(30)
                open_page(driver)
                for kind in ("본문", "키워드·별점"):
                    if kind not in done:
                        collect_section(driver, kind, store, args)
                        done.add(kind)
                state = "수집 종료: 저장 완료(상세 완료 범위는 목록 상태 확인)"
                break
            except AccessBlocked as exc:
                state = "접근 제한으로 부분 수집 저장: " + str(exc)
                logging.error(state)
                store.save(state)
                logging.error("제한 안내가 해제된 뒤 같은 폴더에서 전체 코드 셀을 다시 실행하세요. checkpoint.sqlite3는 보존하세요.")
                break
            except Exception as exc:
                # 대기/클릭 실패 시 현재 화면에 제한 안내가 있는지 먼저 확인.
                detected_block = False
                if driver is not None:
                    try:
                        check_access(driver)
                    except AccessBlocked:
                        detected_block = True
                    except Exception:
                        pass
                if detected_block:
                    state = "접근 제한으로 부분 수집 저장(재시도하지 않음)"
                    logging.error(state)
                    store.save(state)
                    break
                state = "오류로 부분 수집 저장: " + repr(exc)
                logging.exception("수집 오류 — 현재 데이터 보존")
                store.save(state)
                retryable = isinstance(exc, transient_errors) or (
                    isinstance(exc, WebDriverException) and re.search(
                        r"disconnected|chrome not reachable|ERR_CONNECTION|ERR_TIMED_OUT|ERR_NETWORK_CHANGED",
                        str(exc), re.I))
                if not retryable:
                    logging.error("자동 재시도 대상이 아닌 오류입니다. collection.log를 확인하세요.")
                    break
                if attempt >= max(0, args.restarts):
                    break
                delay = min(60 * (2 ** attempt), 300)
                logging.warning("%d/%d 재시작: %d초 후 저장한 행을 보존하고 최신순부터 재탐색합니다.", attempt + 1, args.restarts, delay)
                if driver is not None:
                    try:
                        driver.quit()
                    except Exception:
                        pass
                    driver = None
                store.save("일시적 오류 후 자동 재시작 대기")
                # 사용자의 응답을 기다리지 않습니다.
                time.sleep(delay)
            finally:
                if driver is not None:
                    try:
                        driver.quit()
                    except Exception:
                        pass
                    driver = None
    except KeyboardInterrupt:
        state = "사용자 중단: 현재까지 CSV 저장"
        logging.warning(state)
    except Exception as exc:
        state = "실행 오류: " + repr(exc)
        logging.exception("실행을 계속할 수 없습니다")
    finally:
        try:
            count = store.save(state)
            logging.info("최종 상태: %s | 저장 %d개 | %s", state, count, store.state["CSV파일"])
        except Exception:
            logging.exception("CSV 저장 실패. checkpoint.sqlite3를 지우지 말고 --export-only로 복원하세요.")
        finally:
            store.db.close()
            prevent_sleep(False)


if __name__ == "__main__":
    main()


2026-10-23 19:53:49,331 INFO 저장 폴더: C:\Users\user\everland\everland_output | 기존 기간 내 리뷰: 582
2026-10-23 19:53:55,526 INFO 최신순 정렬 확인. 방문일 범위에 맞는 리뷰를 저장합니다.
2026-10-23 19:54:05,884 INFO 본문 | 확인 10개 | 기간 내 CSV 582개 저장 | 현재 방문일 2026-10-14
2026-10-23 19:54:28,691 INFO 본문 | 확인 20개 | 기간 내 CSV 582개 저장 | 현재 방문일 2026-10-10
2026-10-23 19:54:50,933 INFO 본문 | 확인 30개 | 기간 내 CSV 582개 저장 | 현재 방문일 2026-10-09
2026-10-23 19:55:13,022 INFO 본문 | 확인 40개 | 기간 내 CSV 582개 저장 | 현재 방문일 2026-10-04
2026-10-23 19:55:35,454 INFO 본문 | 확인 50개 | 기간 내 CSV 582개 저장 | 현재 방문일 2026-10-03
2026-10-23 19:55:58,079 INFO 본문 | 확인 60개 | 기간 내 CSV 582개 저장 | 현재 방문일 2026-10-02
2026-10-23 19:56:20,722 INFO 본문 | 확인 70개 | 기간 내 CSV 582개 저장 | 현재 방문일 2026-09-28
2026-10-23 19:56:43,321 INFO 본문 | 확인 80개 | 기간 내 CSV 582개 저장 | 현재 방문일 2026-09-27
2026-10-23 19:57:06,062 INFO 본문 | 확인 90개 | 기간 내 CSV 582개 저장 | 현재 방문일 2026-09-26
2026-10-23 19:57:28,814 INFO 본문 | 확인 100개 | 기간 내 CSV 582개 저장 | 현재 방문일 2026-09-25
2026-10-23 19:57:50,656 INFO 본문 | 확인 110개 | 

In [3]:
from pathlib import Path
import os

폴더 = Path(globals().get("BASE", Path.cwd())) / "everland_output"
print("저장 위치:", 폴더)

if 폴더.is_dir():
    os.startfile(str(폴더))
else:
    print("폴더를 찾지 못했습니다. 위에 출력된 저장 위치를 보내주세요.")

저장 위치: C:\Users\user\everland\everland_output


In [13]:
# -*- coding: utf-8 -*-
"""에버랜드 방문자 리뷰 — 최신순, 방문일 2024-01-01 ~ 실행일(한국 시간). [세션 분할/이어받기 버전]

실행 (주피터):   %run everland_reviews_latest.py
인자 지정:       %run everland_reviews_latest.py --max-new 300 --delay 20
터미널:          python everland_reviews_latest.py --max-new 300
CSV만 복원:      python everland_reviews_latest.py --export-only
진행 초기화:     python everland_reviews_latest.py --reset-progress   (저장된 리뷰는 유지)

이번 버전의 변경점 (이전 코드 대비)
1) 이어받기: 목록별 '확인한 카드 수'를 checkpoint.sqlite3 의 progress 표에 저장합니다.
   다음 실행 때 그 위치까지는 파싱·저장·키워드 펼치기 없이 "더보기"만 눌러 빠르게 이동하고,
   그 이후 카드부터 수집합니다. (이미 끝난 목록은 건너뜁니다.)
   ※ 네이버 목록은 '더보기'를 눌러야만 아래가 열리므로, 이동 구간의 '더보기 클릭 횟수' 자체는
     줄일 수 없습니다. 줄어드는 것은 파싱/저장/펼치기 작업과 대기시간입니다.
2) 요청 간격: 수집 구간 기본 20초(+랜덤 최대 40%), 이동 구간 기본 12초(+랜덤).
   N번 클릭마다 60~90초 긴 휴식을 넣습니다.
3) 세션 분할: 한 번 실행에 새로 수집할 카드 수(--max-new, 기본 300)와 총 클릭 수
   (--max-clicks, 기본 90)에 상한을 둡니다. 상한에 닿으면 정상 종료하고 다음 실행에서 이어갑니다.
4) 날짜 경계: 방문일이 시작일보다 오래된 묶음이 연속 5개(약 50개)면 해당 목록을 종료합니다.
   끄려면 --no-boundary-stop.
5) 접근 제한/인증 화면은 우회하지 않고 그 자리에서 저장 후 종료합니다.

결과는 이 파일 옆 everland_output 폴더:
 everland_reviews.csv (9개 열, 방문일 내림차순, UTF-8 BOM) / checkpoint.sqlite3 /
 review_status.json / collection.log / date_unresolved.csv
아이디=공개 닉네임, 결제유형=화면의 '인증 수단', 재방문=화면의 'N번째 방문' 원문입니다.
화면 선택자 확인일: 2026-10-22. 네이버 화면 개편 시 수정 필요. 이 버전의 새 로직은 실사이트 실행 검증 전입니다.
"""

import argparse
import csv
import ctypes
import hashlib
import json
import logging
import os
import random
import re
import sqlite3
import sys
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

KST = timezone(timedelta(hours=9))
URL = "https://map.naver.com/p/entry/place/11574457?placePath=%2Freview"
COLUMNS = ["아이디", "날짜", "게시글", "평점", "결제유형", "방문유형", "동행자", "재방문", "키워드 리뷰"]
try:
    BASE = Path(__file__).resolve().parent
except NameError:  # 주피터에서 직접 실행한 경우
    BASE = Path.cwd()
PURPOSES = {"데이트", "가족모임", "나들이", "친목", "여행", "일상", "기념일", "회식", "비즈니스", "친목모임", "단체모임"}
COMPANIONS = {"혼자", "아이", "친구", "부모님", "연인・배우자", "연인·배우자", "연인", "배우자", "친척・형제자매", "친척·형제자매", "동료", "지인", "반려동물"}
FINISHED = ("complete", "boundary")  # 다시 수집하지 않는 목록 상태


def now():
    return datetime.now(KST).isoformat(timespec="seconds")


def visit_date(raw):
    """전체 연도가 있는 표기만 사용. 10.21 같은 표기에서 연도를 추정하지 않습니다."""
    for pattern in (r"(?<!\d)(20\d{2})\s*년\s*(\d{1,2})\s*월\s*(\d{1,2})\s*일",
                    r"(?<!\d)(20\d{2})[./-](\d{1,2})[./-](\d{1,2})(?!\d)"):
        m = re.search(pattern, raw or "")
        if m:
            try:
                return date(*map(int, m.groups())).isoformat()
            except ValueError:
                return ""
    return ""


def parse(raw):
    purposes, companions, unknown = [], [], []
    for item in raw.get("visit", []):
        item = item.strip()
        if not item or item.startswith(("대기 시간", "예약 후", "예약 없이")):
            continue
        tokens = [x.strip() for x in item.split(",") if x.strip()]
        if tokens and all(x in PURPOSES for x in tokens):
            purposes.extend(tokens)
        elif tokens and all(x in COMPANIONS for x in tokens):
            companions.extend(tokens)
        else:
            unknown.append(item)
    rating = ""
    m = re.search(r"(?:별점\s*)?([0-5](?:\.\d+)?)\s*(?:점)?$", raw.get("rating", "").strip())
    if m and 0 <= float(m.group(1)) <= 5:
        rating = m.group(1)
    repeat = re.search(r"\d+\s*번째\s*방문", raw.get("meta", ""))
    row = dict(zip(COLUMNS, [
        raw.get("nickname", "").strip(), visit_date(raw.get("date", "")),
        raw.get("content", "").strip(), rating, raw.get("verification", "").strip(),
        " | ".join(dict.fromkeys(purposes)), " | ".join(dict.fromkeys(companions)),
        repeat.group(0) if repeat else "", " | ".join(dict.fromkeys(raw.get("keywords", [])))
    ]))
    return row, unknown


def atomic_text(path, text):
    temp = path.with_name(path.name + ".tmp")
    with temp.open("w", encoding="utf-8") as f:
        f.write(text)
        f.flush()
        os.fsync(f.fileno())
    os.replace(temp, path)


def write_csv(path, rows, columns=COLUMNS):
    """기존 CSV를 먼저 지우지 않음. 새 파일을 모두 쓴 다음 교체."""
    temp = path.with_name(path.name + ".tmp")
    with temp.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
        f.flush()
        os.fsync(f.fileno())
    try:
        os.replace(temp, path)
        return path
    except PermissionError:
        backup = path.with_name(path.stem + "_backup.csv")
        try:
            os.replace(temp, backup)
        except PermissionError:
            backup = path.with_name(path.stem + "_backup_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f") + ".csv")
            os.replace(temp, backup)
        logging.warning("CSV가 열려 있어 대체 파일에 저장: %s", backup)
        return backup


class Store:
    def __init__(self, folder, start, end):
        self.folder, self.start, self.end = Path(folder), start, end
        self.folder.mkdir(parents=True, exist_ok=True)
        self.db = sqlite3.connect(self.folder / "checkpoint.sqlite3")
        self.db.execute("PRAGMA synchronous=FULL")
        self.db.execute("CREATE TABLE IF NOT EXISTS reviews (uid TEXT PRIMARY KEY, visit TEXT, row_json TEXT, raw_json TEXT, unknown TEXT)")
        # 목록별 이어받기 위치: cards=확인 완료한 카드 수, status=collecting/complete/boundary
        self.db.execute("CREATE TABLE IF NOT EXISTS progress (kind TEXT PRIMARY KEY, cards INTEGER, status TEXT, updated TEXT)")
        self.db.commit()
        self.state = {"시작": now(), "시작일": start, "종료일": end, "상태": "실행 준비",
                      "목록": {}, "전체수집보장": False,
                      "안내": "공개 화면에서 확인한 리뷰만 수집. 비공개/삭제/미제공 리뷰는 확인 불가. 동일 프로필·날짜·본문·선택정보의 동일 행은 중복으로 처리."}

    # ---- 이어받기 위치 ----
    def get_progress(self, kind):
        r = self.db.execute("SELECT cards, status FROM progress WHERE kind=?", (kind,)).fetchone()
        return (r[0], r[1]) if r else (0, "new")

    def set_progress(self, kind, cards, status="collecting"):
        with self.db:
            self.db.execute("INSERT OR REPLACE INTO progress VALUES(?,?,?,?)", (kind, cards, status, now()))

    def reset_progress(self):
        with self.db:
            self.db.execute("DELETE FROM progress")

    # ---- 리뷰 저장 ----
    def add(self, records):
        with self.db:
            for raw in records:
                row, unknown = parse(raw)
                identity = [raw.get("profile", ""), row, raw.get("date", "")]
                uid = hashlib.sha256(json.dumps(identity, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
                self.db.execute("INSERT OR REPLACE INTO reviews VALUES(?,?,?,?,?)", (
                    uid, row["날짜"], json.dumps(row, ensure_ascii=False),
                    json.dumps(raw, ensure_ascii=False), json.dumps(unknown, ensure_ascii=False)))

    def save(self, state=None):
        if state:
            self.state["상태"] = state
        rows = [json.loads(x[0]) for x in self.db.execute(
            "SELECT row_json FROM reviews WHERE visit >= ? AND visit <= ? ORDER BY visit DESC, rowid ASC",
            (self.start, self.end))]
        actual = write_csv(self.folder / "everland_reviews.csv", rows)
        unresolved = []
        for row_json, raw_json in self.db.execute("SELECT row_json, raw_json FROM reviews WHERE visit = ''"):
            row, raw = json.loads(row_json), json.loads(raw_json)
            row["날짜원문"] = raw.get("date", "")
            unresolved.append(row)
        write_csv(self.folder / "date_unresolved.csv", unresolved, COLUMNS + ["날짜원문"])
        self.state.update({"마지막저장": now(), "CSV파일": str(actual.resolve()), "저장건수": len(rows),
                           "날짜확인필요": len(unresolved),
                           "선택항목확인필요": self.db.execute("SELECT count(*) FROM reviews WHERE unknown != '[]'").fetchone()[0],
                           "가장최근방문일": rows[0]["날짜"] if rows else None,
                           "가장오래된방문일": rows[-1]["날짜"] if rows else None,
                           "이어받기위치": {k: {"cards": c, "status": s} for k, c, s in
                                      self.db.execute("SELECT kind, cards, status FROM progress")}})
        atomic_text(self.folder / "review_status.json", json.dumps(self.state, ensure_ascii=False, indent=2))
        return len(rows)


# 2026-10-22 실제 공개 리뷰 DOM에서 확인한 선택자.
SECTION_JS = """
const kind = arguments[0];
const section = kind === '본문'
 ? document.querySelector('#_review_list')?.closest('.place_section')
 : Array.from(document.querySelectorAll('.place_section')).find(x =>
     x.querySelector('h2')?.textContent.includes('키워드·별점 리뷰'));
"""
SNAPSHOT_JS = SECTION_JS + """
if (!section) return {missing:true, count:0, records:[], more:false};
const all = Array.from(section.querySelectorAll('li.place_apply_pui'));
function txt(el) { return (el?.textContent || '').trim(); }
function contentText(el) {
  if (!el) return '';
  function walk(n) {
    if (n.nodeType === 3) return n.nodeValue;
    if (n.nodeName === 'BR') return '\\n';
    return Array.from(n.childNodes).map(walk).join('');
  }
  return walk(el).trim();
}
return {missing:false, count:all.length,
  more:!!section.querySelector('[data-nlog-area="plc_rrv.more"], [data-nlog-area="plc_rrv.starmore"]'),
  records:all.slice(arguments[1]).map(card => {
    const meta = card.querySelector('.pui__QKE5Pr');
    const dateNode = meta?.querySelector('time');
    const parts = Array.from(meta?.children || []);
    const verify = parts.find(n => n.textContent.includes('인증 수단'));
    return {
      nickname:txt(card.querySelector('.pui__NMi-Dp')),
      profile:card.querySelector('[data-pui-click-code="profile"]')?.getAttribute('href') || '',
      date:txt(dateNode?.parentElement), meta:txt(meta),
      verification:txt(verify).replace('인증 수단','').trim(),
      rating:txt(card.querySelector('.pui__6abRMf')),
      content:contentText(card.querySelector('[data-pui-click-code="rvshowmore"], [data-pui-click-code="rvshowless"]')),
      visit:Array.from(card.querySelectorAll('[data-pui-click-code="visitkeywords"] > span')).map(txt),
      keywords:Array.from(card.querySelectorAll('li.pui__jhpEyP')).map(txt)
    };
  })};
"""
MORE_JS = SECTION_JS + "return section?.querySelector('[data-nlog-area=\"plc_rrv.more\"], [data-nlog-area=\"plc_rrv.starmore\"]');"


class AccessBlocked(RuntimeError):
    pass


class RetryableReviewError(RuntimeError):
    """리뷰 응답 지연처럼 저장 후 재시도할 수 있는 오류."""


class SessionLimit(Exception):
    """이번 실행의 수집량/클릭 상한 도달. 오류가 아니라 정상 종료입니다."""


class Session:
    """한 번 실행 동안의 사용량 집계(재시작해도 유지)."""
    def __init__(self):
        self.new_cards = 0
        self.clicks = 0


def restriction_message(text):
    text = re.sub(r"\s+", " ", text or "").strip()
    patterns = (
        r"자동\s*입력\s*방지|비정상적인\s*(?:접근|요청|트래픽)|접근이?\s*제한",
        r"자동화된\s*요청|로봇이\s*아닙니다|Verify you are human|Access Denied",
        r"요청이?\s*차단|Too Many Requests|이용이\s*(?:일시적으로\s*)?제한",
        r"(?:IP|아이피).{0,100}(?:차단|제한|비정상|너무\s*많은)",
        r"(?:차단|제한|비정상|너무\s*많은).{0,100}(?:IP|아이피)",
    )
    return any(re.search(p, text, re.I) for p in patterns)


def check_access(driver):
    text = driver.execute_script("""
    const cards = document.querySelector('li.place_apply_pui');
    const alert = Array.from(document.querySelectorAll('[role="alert"], [role="dialog"], h1, h2, h3, p'))
      .filter(x=>!x.closest('li.place_apply_pui') && !x.querySelector('li.place_apply_pui'))
      .map(x=>x.innerText).join(' ');
    return cards ? alert : document.body.innerText;
    """)
    if restriction_message(text):
        raise AccessBlocked("접근 제한/인증 화면 감지: 우회하지 않고 현재 결과를 저장합니다.")


def wait_for(driver, predicate, seconds):
    from selenium.webdriver.support.ui import WebDriverWait

    def checked(current):
        check_access(current)
        return predicate(current)
    return WebDriverWait(driver, seconds, poll_frequency=1).until(checked)


def click(driver, element):
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", element)
    element.click()


def open_page(driver):
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support import expected_conditions as EC
    driver.get(URL)
    wait_for(driver, EC.frame_to_be_available_and_switch_to_it((By.ID, "entryIframe")), 45)
    check_access(driver)
    recent = wait_for(driver, EC.element_to_be_clickable((By.XPATH, "//a[normalize-space(.)='최신순']")), 45)
    click(driver, recent)
    wait_for(driver, lambda d: "reviewSort=recent" in d.execute_script("return window.location.search"), 45)
    wait_for(driver, lambda d: d.find_elements(By.CSS_SELECTOR, "#_review_list li.place_apply_pui"), 45)
    logging.info("최신순 정렬 확인.")


def pause(args, session, fast=False):
    """더보기 클릭 사이 대기. 랜덤 지터 + N클릭마다 긴 휴식."""
    base = args.ff_delay if fast else args.delay
    time.sleep(max(1.5, random.uniform(base, base * 1.4)))
    session.clicks += 1
    if args.rest_every and session.clicks % args.rest_every == 0:
        rest = random.uniform(60, 90)
        logging.info("휴식 %.0f초 (이번 실행 클릭 %d회)", rest, session.clicks)
        time.sleep(rest)


def check_limits(session, args):
    if session.clicks >= args.max_clicks:
        raise SessionLimit("이번 실행 클릭 상한(%d) 도달" % args.max_clicks)
    if session.new_cards >= args.max_new:
        raise SessionLimit("이번 실행 신규 수집 상한(%d개) 도달" % args.max_new)


def click_more_and_wait(driver, kind, before):
    """더보기 1회 클릭 후 카드 수 증가를 기다립니다. 반환: 'grew' | 'end'."""
    from selenium.common.exceptions import TimeoutException
    more = driver.execute_script(MORE_JS, kind)
    if more is None:
        time.sleep(3)
        check_access(driver)
        snap = driver.execute_script(SNAPSHOT_JS, kind, before)
        if snap["count"] > before or snap["more"]:
            return "grew"
        return "end"
    click(driver, more)
    try:
        wait_for(driver, lambda d: d.execute_script(SNAPSHOT_JS, kind, before)["count"] > before, 45)
    except TimeoutException:
        check_access(driver)
        try:
            wait_for(driver, lambda d: d.execute_script(SNAPSHOT_JS, kind, before)["count"] > before, 30)
        except TimeoutException:
            final = driver.execute_script(SNAPSHOT_JS, kind, before)
            if not final["more"] and final["count"] >= before:
                return "end"
            raise RetryableReviewError("더보기 뒤 새 리뷰가 75초 동안 추가되지 않았습니다.")
    return "grew"


def fast_forward(driver, kind, target, store, args, session):
    """저장 위치(target 카드)까지 파싱/저장 없이 더보기만 눌러 이동. 반환: (현재 카드 수, 끝 도달 여부)"""
    snap = driver.execute_script(SNAPSHOT_JS, kind, 10 ** 9)
    count = snap["count"]
    while count < target:
        check_limits(session, args)
        pause(args, session, fast=True)
        result = click_more_and_wait(driver, kind, count)
        count = driver.execute_script(SNAPSHOT_JS, kind, 10 ** 9)["count"]
        logging.info("%s | 이어받기 이동 중 %d / %d", kind, count, target)
        if result == "end":
            return count, True
    return count, False


def collect_section(driver, kind, store, args, session):
    from selenium.webdriver.common.by import By
    resume, status = store.get_progress(kind)
    if status in FINISHED:
        logging.info("%s | 이전 실행에서 끝남(%s) → 건너뜀", kind, status)
        return status
    if kind == "키워드·별점":
        toggles = driver.find_elements(By.CSS_SELECTOR, '[data-nlog-area="plc_rrv.star"]')
        if not toggles:
            raise RuntimeError("키워드·별점 리뷰 섹션을 확인하지 못했습니다. 완료로 처리하지 않습니다.")
        if toggles[0].get_attribute("aria-expanded") != "true":
            click(driver, toggles[0])
        wait_for(driver, lambda d: d.execute_script(SNAPSHOT_JS, kind, 0)["count"] > 0, 45)

    offset = 0
    if resume > 0:
        logging.info("%s | 이어받기: %d개 카드 위치까지 이동합니다.", kind, resume)
        count, ended = fast_forward(driver, kind, resume, store, args, session)
        offset = min(resume, count)  # 저장 위치 이후 카드부터 수집
        if ended and count <= resume:
            store.set_progress(kind, count, "complete")
            store.state["목록"][kind] = {"상태": "화면 목록 끝까지 확인", "확인카드": count}
            store.save()
            return "complete"

    batch, old_streak, prev_date, monotonic = 0, 0, None, True
    while True:
        check_access(driver)
        snap = driver.execute_script(SNAPSHOT_JS, kind, offset)
        if snap["missing"]:
            raise RuntimeError(kind + " 리뷰 목록을 찾지 못했습니다.")
        if snap["count"] < offset:
            raise RetryableReviewError("목록 길이가 줄었습니다. 재탐색이 필요합니다.")
        records = snap["records"]
        if records:
            section = driver.execute_script(SECTION_JS + "return section;", kind)
            cards = section.find_elements(By.CSS_SELECTOR, "li.place_apply_pui")[offset:]
            expanded = False
            for card in cards:
                ex = card.find_elements(By.CSS_SELECTOR, '[data-pui-click-code="keywordmore"]')
                if ex and ex[0].is_displayed():
                    click(driver, ex[0])
                    time.sleep(1)
                    expanded = True
            if expanded:
                records = driver.execute_script(SNAPSHOT_JS, kind, offset)["records"]
            store.add(records)  # CSV보다 먼저 디스크에 커밋
            dates = [visit_date(r.get("date", "")) for r in records]
            for d in dates:
                if not d:
                    monotonic = False
                else:
                    if prev_date and d > prev_date:
                        monotonic = False
                    prev_date = d
            old_streak = old_streak + 1 if dates and all(d and d < args.start for d in dates) else 0
            session.new_cards += len(records)
            offset, batch = snap["count"], batch + 1
            store.set_progress(kind, offset, "collecting")  # 이어받기 위치 즉시 기록
            store.state["목록"][kind] = {"상태": "수집 중", "확인카드": offset, "마지막방문일": dates[-1]}
            saved = store.save("수집 중")
            logging.info("%s | 확인 %d개 | CSV %d개 | 현재 방문일 %s | 이번 실행 신규 %d/%d",
                         kind, offset, saved, dates[-1] or "확인 불가", session.new_cards, args.max_new)
            if args.boundary_stop and monotonic and old_streak >= args.boundary_batches:
                store.set_progress(kind, offset, "boundary")
                store.state["목록"][kind]["상태"] = "날짜 경계로 종료(누락 여부 미확인)"
                store.save()
                return "boundary"
        check_limits(session, args)
        if driver.execute_script(MORE_JS, kind) is None:
            time.sleep(3)
            check_access(driver)
            after = driver.execute_script(SNAPSHOT_JS, kind, offset)
            if after["count"] > offset or after["more"]:
                continue
            if after["missing"] or after["count"] == 0:
                raise RuntimeError("목록이 비어 전체 확인 여부를 판단할 수 없습니다.")
            store.set_progress(kind, offset, "complete")
            store.state["목록"][kind]["상태"] = "화면 목록 끝까지 확인"
            store.save()
            return "complete"
        pause(args, session)
        result = click_more_and_wait(driver, kind, offset)
        if result == "end":
            final = driver.execute_script(SNAPSHOT_JS, kind, offset)
            if final["records"]:
                continue  # 마지막으로 늘어난 카드를 먼저 처리
            store.set_progress(kind, offset, "complete")
            store.state["목록"][kind]["상태"] = "화면 목록 끝까지 확인"
            store.save()
            return "complete"


def prevent_sleep(enable):
    if sys.platform == "win32":
        try:
            result = ctypes.windll.kernel32.SetThreadExecutionState(0x80000001 if enable else 0x80000000)
            if enable and not result:
                logging.warning("절전 억제 실패: Windows 전원 설정에서 절전을 직접 꺼 주세요.")
        except Exception:
            logging.warning("절전 억제를 적용하지 못했습니다.")


def build_parser():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--start", default="2024-01-01")
    p.add_argument("--end", default=datetime.now(KST).date().isoformat())
    p.add_argument("--output", type=Path, default=BASE / "everland_output")
    p.add_argument("--delay", type=float, default=20.0, help="수집 구간 더보기 간격(초). 실제는 이 값~1.4배 랜덤")
    p.add_argument("--ff-delay", type=float, default=12.0, help="이어받기 이동 구간 더보기 간격(초)")
    p.add_argument("--max-new", type=int, default=300, help="이번 실행에서 새로 수집할 카드 상한")
    p.add_argument("--max-clicks", type=int, default=90, help="이번 실행 더보기 클릭 총상한(이동 구간 포함)")
    p.add_argument("--rest-every", type=int, default=15, help="N클릭마다 60~90초 휴식(0이면 끔)")
    p.add_argument("--restarts", type=int, default=3)
    p.add_argument("--boundary-batches", type=int, default=5, help="시작일보다 오래된 묶음이 연속 N개면 종료")
    p.add_argument("--no-boundary-stop", dest="boundary_stop", action="store_false", help="날짜 경계 조기 종료 끄기")
    p.add_argument("--export-only", action="store_true", help="브라우저 없이 체크포인트에서 CSV 복원")
    p.add_argument("--reset-progress", action="store_true", help="이어받기 위치만 초기화(저장된 리뷰는 유지)")
    p.set_defaults(boundary_stop=True)
    return p


def main(argv=None):
    parser = build_parser()
    args, _ = parser.parse_known_args(argv)  # 주피터의 -f 인자 등은 무시
    if date.fromisoformat(args.start) > date.fromisoformat(args.end):
        parser.error("시작일은 종료일보다 늦을 수 없습니다.")
    args.output.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",
                        handlers=[logging.StreamHandler(), logging.FileHandler(args.output / "collection.log", encoding="utf-8")], force=True)
    store, driver, session = Store(args.output, args.start, args.end), None, Session()
    state = "시작 전"
    prevent_sleep(True)
    try:
        if args.reset_progress:
            store.reset_progress()
            logging.info("이어받기 위치를 초기화했습니다(저장된 리뷰는 유지).")
        count = store.save("기존 수집분 복원")
        logging.info("저장 폴더: %s | 기존 기간 내 리뷰: %d", args.output.resolve(), count)
        if args.export_only:
            state = "체크포인트에서 CSV 복원 완료(수집 실행 아님)"
            return
        from selenium import webdriver
        from selenium.webdriver.chrome.options import Options
        from selenium.common.exceptions import (
            TimeoutException, StaleElementReferenceException,
            ElementClickInterceptedException, InvalidSessionIdException,
            NoSuchWindowException, WebDriverException,
        )
        transient_errors = (RetryableReviewError, TimeoutException, StaleElementReferenceException,
                            ElementClickInterceptedException, InvalidSessionIdException, NoSuchWindowException)
        for attempt in range(max(0, args.restarts) + 1):
            try:
                options = Options()
                options.add_argument("--lang=ko-KR")
                options.add_argument("--window-size=1440,1000")
                driver = webdriver.Chrome(options=options)
                driver.set_page_load_timeout(60)
                driver.set_script_timeout(30)
                open_page(driver)
                results = {}
                for kind in ("본문", "키워드·별점"):
                    results[kind] = collect_section(driver, kind, store, args, session)
                state = "두 목록 모두 종료: " + ", ".join("%s=%s" % kv for kv in results.items())
                break
            except SessionLimit as exc:
                state = "세션 분할 정상 종료: %s. 시간을 두고 같은 명령으로 다시 실행하면 이어서 수집합니다." % exc
                logging.info(state)
                break
            except AccessBlocked as exc:
                state = "접근 제한으로 부분 수집 저장: " + str(exc)
                logging.error(state)
                store.save(state)
                logging.error("바로 재실행하지 말고 충분히(수 시간~하루) 쉰 뒤, 같은 폴더에서 --delay를 더 늘려 다시 실행하세요. checkpoint.sqlite3는 보존하세요.")
                break
            except Exception as exc:
                detected_block = False
                if driver is not None:
                    try:
                        check_access(driver)
                    except AccessBlocked:
                        detected_block = True
                    except Exception:
                        pass
                if detected_block:
                    state = "접근 제한으로 부분 수집 저장(재시도하지 않음)"
                    logging.error(state)
                    store.save(state)
                    break
                state = "오류로 부분 수집 저장: " + repr(exc)
                logging.exception("수집 오류 — 현재 데이터 보존")
                store.save(state)
                retryable = isinstance(exc, transient_errors) or (
                    isinstance(exc, WebDriverException) and re.search(
                        r"disconnected|chrome not reachable|ERR_CONNECTION|ERR_TIMED_OUT|ERR_NETWORK_CHANGED",
                        str(exc), re.I))
                if not retryable:
                    logging.error("자동 재시도 대상이 아닌 오류입니다. collection.log를 확인하세요.")
                    break
                if attempt >= max(0, args.restarts):
                    break
                delay = min(60 * (2 ** attempt), 300)
                logging.warning("%d/%d 재시작: %d초 후 저장 위치부터 이어갑니다.", attempt + 1, args.restarts, delay)
                if driver is not None:
                    try:
                        driver.quit()
                    except Exception:
                        pass
                    driver = None
                store.save("일시적 오류 후 자동 재시작 대기")
                time.sleep(delay)
            finally:
                if driver is not None:
                    try:
                        driver.quit()
                    except Exception:
                        pass
                    driver = None
    except KeyboardInterrupt:
        state = "사용자 중단: 현재까지 CSV 저장"
        logging.warning(state)
    except Exception as exc:
        state = "실행 오류: " + repr(exc)
        logging.exception("실행을 계속할 수 없습니다")
    finally:
        try:
            count = store.save(state)
            logging.info("최종 상태: %s | 저장 %d개 | %s", state, count, store.state.get("CSV파일"))
        except Exception:
            logging.exception("CSV 저장 실패. checkpoint.sqlite3를 지우지 말고 --export-only로 복원하세요.")
        finally:
            store.db.close()
            prevent_sleep(False)


if __name__ == "__main__":
    main()

2026-10-23 22:17:15,274 INFO 저장 폴더: C:\Users\user\everland\everland_output | 기존 기간 내 리뷰: 582
2026-10-23 22:17:21,794 INFO 최신순 정렬 확인.
2026-10-23 22:17:21,796 INFO 본문 | 이어받기: 570개 카드 위치까지 이동합니다.
2026-10-23 22:17:38,576 INFO 본문 | 이어받기 이동 중 20 / 570
2026-10-23 22:17:55,796 INFO 본문 | 이어받기 이동 중 30 / 570
2026-10-23 22:18:11,603 INFO 본문 | 이어받기 이동 중 40 / 570
2026-10-23 22:18:25,234 INFO 본문 | 이어받기 이동 중 50 / 570
2026-10-23 22:18:39,418 INFO 본문 | 이어받기 이동 중 60 / 570
2026-10-23 22:18:56,428 INFO 본문 | 이어받기 이동 중 70 / 570
2026-10-23 22:19:09,753 INFO 본문 | 이어받기 이동 중 80 / 570
2026-10-23 22:19:23,312 INFO 본문 | 이어받기 이동 중 90 / 570
2026-10-23 22:19:40,899 INFO 본문 | 이어받기 이동 중 100 / 570
2026-10-23 22:19:55,409 INFO 본문 | 이어받기 이동 중 110 / 570
2026-10-23 22:20:10,997 INFO 본문 | 이어받기 이동 중 120 / 570
2026-10-23 22:20:28,096 INFO 본문 | 이어받기 이동 중 130 / 570
2026-10-23 22:20:46,553 INFO 본문 | 이어받기 이동 중 140 / 570
2026-10-23 22:21:03,082 INFO 본문 | 이어받기 이동 중 150 / 570
2026-10-23 22:21:15,312 INFO 휴식 88초 (이번 실행 클릭 15회)
2026-10-